# Fold, seed and evaluation cost (Reviewer 2, comment 4 and minor comment 1)

**Definitions** (the same in every notebook, see the docstring of `analysis.py`):

- *improvement*: % change in CPU cycles of the returned patch on the validation inputs (negative = faster).
  A patch that fails the correctness oracle on the validation inputs counts as 0% (the original program is kept).
- *gap*: for each run, validation change minus training change, in percentage points (positive = the patch
  generalises worse than it trained). Aggregated with the median of the runs.
- *cell*: (software, algorithm, budget), 15 runs = 5 folds × 3 seeds. Each run has 1000 evaluations; the budgets
  100, 250, 500 and 750 are checkpoints of the same run.
- *ranking*: per software, the median of each cell, averaged over the five budgets and ranked; rank sum over the
  four systems (lower = better).

In [1]:
import pathlib
import sys

import numpy as np
import pandas as pd
from IPython.display import display

sys.path.insert(0, str(pathlib.Path('..', 'revision_analysis').resolve()))
import analysis as A

RESULTS = 'results.csv'           # one row per run and budget (checkpoint)
OUT = pathlib.Path('analysis')    # tables written by analysis.py
REFRESH = False                   # True: recompute every table from results.csv
if REFRESH or not (OUT / 'cells.csv').exists():
    A.run(RESULTS, OUT, 'zero')
df = A.load(RESULTS, 'zero')
cells = pd.read_csv(OUT / 'cells.csv')
SW = {'minisat_hack': 'MiniSAT-hack', 'sat4j': 'Sat4J', 'weka': 'Weka', 'pngOptim': 'OptiPNG'}
pd.set_option('display.precision', 2)
pd.set_option('display.max_rows', 300)
pd.set_option('display.width', 200)

## Share of the variance due to the fold

For every cell, $\eta^2$ of the fold: the share of the variance of the 15 runs explained by the fold (the rest is due to the seed, i.e. the search). With 5 folds of 3 runs, a fold label with no effect gives $\eta^2$ = 4/14 = 0.29 on average. The permutation test shuffles the fold labels 1000 times within the cell.

In [2]:
fv = pd.read_csv(OUT / 'fold_variance.csv')
display(fv.groupby(['metric', 'software']).agg(median_eta2=('eta2_fold', 'median'),
                                               cells_p_below_005=('p_perm', lambda p: (p < 0.05).sum()),
                                               cells=('p_perm', 'size')))
print(fv.groupby('metric').eta2_fold.median().round(2).to_dict())

median_eta2  cells_p_below_005  cells
metric      software                                           
gap         minisat_hack         0.35                  5     20
            pngOptim             0.26                  0     20
            sat4j                0.36                  4     20
            weka                 0.39                  2     20
improvement minisat_hack         0.32                  5     20
            pngOptim             0.24                  1     20
            sat4j                0.31                  1     20
            weka                 0.33                  1     20

{'gap': 0.31, 'improvement': 0.31}


## Candidates evaluated more than once

`repeated_share`: share of the evaluations of a run whose patch had already been evaluated earlier in the same run; every algorithm counts them against its budget. MAGPIE keeps a cache of recently evaluated variants, so only part of them is answered from it (`cached_share`); the rest is run again.

In [3]:
c = cells[cells.group.isin(A.BASE)]
for col in ['repeated_share_mean', 'cached_share_mean']:
    display(c[c.budget == 1000].pivot(index='software', columns='group', values=col).rename_axis(columns=col))
    display(c[c.budget == 1000].groupby('group')[col].mean())

repeated_share_mean,FI,GP,RS,UMDA
software,,,,
minisat_hack,0.06,0.37,0.09,0.62
pngOptim,0.04,0.45,0.05,0.61
sat4j,0.04,0.40,0.05,0.63
weka,0.14,0.44,0.21,0.63


group
FI      0.07
GP      0.41
RS      0.10
UMDA    0.62
Name: repeated_share_mean, dtype: float64

cached_share_mean,FI,GP,RS,UMDA
software,,,,
minisat_hack,0.11,0.43,1.54e-02,0.54
pngOptim,0.03,0.47,7.87e-03,0.53
sat4j,0.03,0.47,7.53e-03,0.55
weka,0.13,0.49,4.71e-02,0.56


group
FI      0.08
GP      0.46
RS      0.02
UMDA    0.55
Name: cached_share_mean, dtype: float64

## Patches that fail the correctness oracle on the validation inputs

In [4]:
display(pd.read_csv(OUT / 'val_fail_rates.csv'))

,software,group,runs,checkpoints,failed_checkpoints,fail_rate,fail_rate_1000,runs_with_failure
0,minisat_hack,ABL-FI-2,15,75,0,0.00,0.00,0
1,minisat_hack,ABL-GP-2,15,75,0,0.00,0.00,0
2,minisat_hack,ABL-UMDA-imp,15,75,0,0.00,0.00,0
3,minisat_hack,ABL-UMDA-k1,15,75,0,0.00,0.00,0
4,minisat_hack,ABL-UMDA-k3,15,75,0,0.00,0.00,0
5,minisat_hack,FI,15,75,0,0.00,0.00,0
6,minisat_hack,GP,15,75,0,0.00,0.00,0
7,minisat_hack,LLM-llama8b,15,75,0,0.00,0.00,0
8,minisat_hack,LLM-qwen7b,15,75,8,0.11,0.13,2
9,minisat_hack,RS,15,75,3,0.04,0.07,1
